In [1]:
import time
import ee
import geemap
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, clear_output
from shapely.geometry import mapping
import os
from pathlib import Path
from datetime import datetime

In [2]:
shapefile_path = Path("C:/Users/julio/OneDrive/GEO/Teste_Automatiza/Perimetro/jus.shp")
download_dir = Path("C:/Users/julio/OneDrive/GEO/Teste_Automatiza/Perimetro")
buffer_distance = 7000  # Em metros

# Autenticação e inicialização do Google Earth Engine
try:
    ee.Initialize()
    print("Google Earth Engine inicializado com sucesso.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine autenticado e inicializado com sucesso.")

Google Earth Engine inicializado com sucesso.


In [3]:
# Autenticação e inicialização
try:
    ee.Initialize()
    print("Google Earth Engine inicializado com sucesso.")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("Google Earth Engine autenticado e inicializado com sucesso.")

# Função para carregar shapefile e aplicar buffer em metros reais
def load_shapefile_with_buffer(shapefile_path, buffer_m):
    print("Tentando carregar o shapefile...")
    if not shapefile_path.exists():
        raise FileNotFoundError(f"Shapefile não encontrado: {shapefile_path}")
    
    gdf = gpd.read_file(shapefile_path)
    print("Shapefile carregado com sucesso.")
    
    utm_crs = gdf.estimate_utm_crs()
    print(f"Reprojetando para CRS métrico: {utm_crs}")
    gdf_utm = gdf.to_crs(utm_crs)

    gdf_utm['geometry'] = gdf_utm.buffer(buffer_m)

    gdf_wgs = gdf_utm.to_crs(epsg=4326)

    combined_geom = gdf_wgs.geometry.unary_union
    geo_json = mapping(combined_geom)
    ee_geom = ee.Geometry(geo_json)
    return ee_geom, gdf_wgs

# Carregar shapefile com buffer
buffered_region, gdf = load_shapefile_with_buffer(shapefile_path, buffer_distance)

# Buscar coleção
def get_sentinel_collection(collection_id, start_date, end_date, region):
    with output:
        output.append_stdout(f"\nBuscando imagens de {start_date} até {end_date}...\n")
    collection = ee.ImageCollection(collection_id) \
        .filterDate(start_date, end_date) \
        .filterBounds(region)
    size = collection.size().getInfo()
    with output:
        output.append_stdout(f"Número de imagens encontradas: {size}\n")
    if size > 0:
        return collection
    else:
        raise Exception("Nenhuma imagem encontrada para o período especificado.")

# Widgets
start_date_widget = widgets.DatePicker(description='Data Inicial:', value=datetime(2016, 9, 1).date())
end_date_widget = widgets.DatePicker(description='Data Final:', value=datetime(2016, 10, 30).date())
submit_button = widgets.Button(description="Buscar Imagens")
image_selector = widgets.SelectMultiple(description='Datas:', options=[])
download_button = widgets.Button(description="Baixar Imagens")
visualize_button = widgets.Button(description="Visualizar Imagem")
output = widgets.Output()

# Atualizar lista de imagens
def update_image_list():
    global collection, image_list, image_dates
    with output:
        clear_output(wait=True)
        output.append_stdout("Atualizando lista de imagens...\n")
    try:
        collection = get_sentinel_collection("COPERNICUS/S2_HARMONIZED", start_date_widget.value.isoformat(), end_date_widget.value.isoformat(), buffered_region)
        image_list = collection.toList(collection.size())
        image_dates = [ee.Image(image_list.get(i)).date().format('YYYY-MM-dd').getInfo() for i in range(collection.size().getInfo())]
        image_selector.options = image_dates
        with output:
            output.append_stdout("Lista de imagens atualizada com sucesso.\n")
    except Exception as e:
        with output:
            output.append_stdout(f"Erro ao buscar imagens: {e}\n")

# Visualizar imagem
def display_image(b):
    date = image_selector.value[0] if image_selector.value else None
    if date is None:
        with output:
            clear_output(wait=True)
            output.append_stdout("Por favor, selecione uma data de imagem primeiro.\n")
        return
    index = image_dates.index(date)
    image = ee.Image(image_list.get(index))
    rgb_bands = ['B4', 'B3', 'B2']
    vis_params = {'min': 0, 'max': 3000, 'bands': rgb_bands}
    Map = geemap.Map()
    Map.centerObject(buffered_region, 10)
    Map.addLayer(image.select(rgb_bands), vis_params, 'Imagem Sentinel-2 RGB')
    Map.add_gdf(gdf, 'Shapefile', style={'color': 'red', 'fillOpacity': 0.1, 'weight': 2})
    with output:
        clear_output(wait=True)
        display(Map)

# Processar e exportar imagens RGB, FC e NDVI
def process_and_download_images(b):
    selected_dates = image_selector.value
    if not selected_dates:
        with output:
            clear_output(wait=True)
            output.append_stdout("Por favor, selecione pelo menos uma data de imagem primeiro.\n")
        return

    with output:
        clear_output(wait=True)
        output.append_stdout(f"Iniciando exportação para {len(selected_dates)} data(s)...\n")

    for date in selected_dates:
        with output:
            output.append_stdout(f"\n📅 Processando imagem: {date}\n")

        image = ee.Image(collection.filterDate(ee.Date(date), ee.Date(date).advance(1, 'day')).first())
        buffer_coords = buffered_region.bounds().getInfo()['coordinates']

        year_folder = download_dir / date[:4]
        year_folder.mkdir(parents=True, exist_ok=True)

        rgb_path = year_folder / f"Sentinel_RGB_{date[5:7]}_{date[:4]}.tif"
        false_color_path = year_folder / f"Sentinel_FC_{date[5:7]}_{date[:4]}.tif"
        ndvi_path = year_folder / f"Sentinel_NDVI_{date[5:7]}_{date[:4]}.tif"

        try:
            with output:
                output.append_stdout(f"➡️ Exportando RGB para: {rgb_path.name}\n")
            geemap.ee_export_image(image.select(['B4', 'B3', 'B2']), filename=str(rgb_path), scale=10, region=buffer_coords, file_per_band=False)
            time.sleep(5)

            with output:
                output.append_stdout(f"➡️ Exportando Falsa Cor para: {false_color_path.name}\n")
            geemap.ee_export_image(image.select(['B8', 'B3', 'B2']), filename=str(false_color_path), scale=10, region=buffer_coords, file_per_band=False)
            time.sleep(5)

            with output:
                output.append_stdout(f"➡️ Calculando e exportando NDVI para: {ndvi_path.name}\n")
            ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
            geemap.ee_export_image(ndvi, filename=str(ndvi_path), scale=10, region=buffer_coords)
            time.sleep(5)

            with output:
                output.append_stdout(f"✅ Concluído: {date}\n")
        except Exception as e:
            with output:
                output.append_stdout(f"❌ Erro ao exportar imagens de {date}: {e}\n")

    with output:
        output.append_stdout("\n✅ Todas as imagens selecionadas foram processadas e salvas.\n")

# Conectar botões
submit_button.on_click(lambda b: update_image_list())
download_button.on_click(process_and_download_images)
visualize_button.on_click(display_image)

# Mostrar interface
display(start_date_widget, end_date_widget, submit_button, image_selector, visualize_button, download_button, output)

Google Earth Engine inicializado com sucesso.
Tentando carregar o shapefile...
Shapefile carregado com sucesso.
Reprojetando para CRS métrico: EPSG:32723


C:\Users\julio\AppData\Local\Temp\ipykernel_17436\3763448004.py:27: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  combined_geom = gdf_wgs.geometry.unary_union


DatePicker(value=datetime.date(2016, 9, 1), description='Data Inicial:', step=1)

DatePicker(value=datetime.date(2016, 10, 30), description='Data Final:', step=1)

Button(description='Buscar Imagens', style=ButtonStyle())

SelectMultiple(description='Datas:', options=(), value=())

Button(description='Visualizar Imagem', style=ButtonStyle())

Button(description='Baixar Imagens', style=ButtonStyle())

Output()